# RNAQwen 35B full training on Colab

Required GPU: A100 40GB or better.

Drive layout:
```text
MyDrive/RNAQwen/
  RNAQwen_code.zip
  assets/
    _fsq_full_pos.pt
    _rfam_hidden.f16
    _rfam_index.pt
    rnaqa_v3_l234.jsonl
```

Run the cells in order. The smoke cell must pass before the full cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

DRIVE = Path('/content/drive/MyDrive/RNAQwen')
ROOT = Path('/content/RNAQwen')
MODEL_DIR = Path('/content/models/Qwen3.6-35B-A3B')
ASSETS = DRIVE / 'assets'
CODE_ZIP = DRIVE / 'RNAQwen_code.zip'
RUN_TAG = 'v3_full'
RUN_DIR_NAME = 'e2e_' + RUN_TAG
DRIVE_RUN = DRIVE / 'runs' / RUN_DIR_NAME

print('GPU:')
subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version',
    '--format=csv,noheader'
], check=False)
print('\nDrive:', DRIVE)
print('Code zip exists:', CODE_ZIP.exists())
print('Assets dir exists:', ASSETS.exists())

free_gb = shutil.disk_usage('/content').free / 2**30
print(f'Local free disk: {free_gb:.1f} GiB')
if free_gb < 100:
    print('WARNING: less than 100 GiB free; model + checkpoints may not fit.')

In [ ]:
%pip install -q transformers==5.9.0 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2 safetensors==0.8.0 sentencepiece==0.2.2

In [ ]:
if not CODE_ZIP.exists():
    raise FileNotFoundError(f'Missing {CODE_ZIP}')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
subprocess.run(['unzip', '-q', str(CODE_ZIP), '-d', str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Code ready:', ROOT)
print(subprocess.run(
    [sys.executable, '-c',
     'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))'],
    text=True, capture_output=True
).stdout.strip())

In [ ]:
DATA_DIR = ROOT / 'data' / 'processed'
DATA_DIR.mkdir(parents=True, exist_ok=True)
required = {
    '_fsq_full_pos.pt': 6_201_932_003,
    '_rfam_hidden.f16': 10_261_060_608,
    '_rfam_index.pt': 829_765,
    'rnaqa_v3_l234.jsonl': 663_240_811,
}

for name, expected_size in required.items():
    src = ASSETS / name
    dst = DATA_DIR / name
    if not src.exists():
        if dst.exists():
            print('Using local asset:', dst)
            continue
        raise FileNotFoundError(f'Missing {src}')
    if src.stat().st_size != expected_size:
        raise RuntimeError(
            f'Incomplete asset {src}: {src.stat().st_size:,} != {expected_size:,}'
        )
    if dst.exists() and dst.stat().st_size == expected_size:
        print('Already staged:', name)
        continue
    print(f'Copying {name}: {src.stat().st_size / 2**30:.2f} GiB')
    shutil.copyfile(src, dst)

print('Assets ready:', DATA_DIR)

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
cfg = MODEL_DIR / 'config.json'
if not cfg.exists():
    print('Downloading Qwen3.6-35B-A3B from ModelScope (about 67 GB)...')
    subprocess.run([
        sys.executable, '-u', str(ROOT / 'scripts/00b_download_modelscope.py'),
        '--repo', 'Qwen/Qwen3.6-35B-A3B',
        '--dest', str(MODEL_DIR),
        '--workers', '4'
    ], cwd=ROOT, check=True)
else:
    print('Model config already staged:', MODEL_DIR)

index_path = MODEL_DIR / 'model.safetensors.index.json'
if not index_path.exists():
    raise FileNotFoundError(index_path)
index = json.loads(index_path.read_text(encoding='utf-8'))
missing = [name for name in set(index['weight_map'].values())
           if not (MODEL_DIR / name).exists()]
if missing:
    raise RuntimeError(f'Model is incomplete, missing {len(missing)} shards: {missing[:3]}')

os.environ['RNAQ_QWEN36_DIR'] = str(MODEL_DIR)
os.environ['RNAQ_THREADS'] = '8'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print('RNAQ_QWEN36_DIR =', os.environ['RNAQ_QWEN36_DIR'])

In [ ]:
common = [
    sys.executable, '-X', 'utf8', '-u', str(ROOT / 'scripts/50_routeC_e2e.py'),
    '--data', str(DATA_DIR / 'rnaqa_v3_l234.jsonl'),
    '--hid2', str(DATA_DIR / '_rfam_hidden.f16'),
    '--hid2-idx', str(DATA_DIR / '_rfam_index.pt'),
    '--fsq', str(ROOT / 'checkpoints/fsq_stream/tokenizer.pt'),
    '--init-code-emb', str(ROOT / 'checkpoints/fsq_stream/code_emb_contentinit.pt'),
    '--base', 'qwen36-35b', '--qwen36-mode', 'int4',
    '--adapter', 'sos', '--lora-r', '8',
    '--token-budget', '512',
]

smoke = common + [
    '--steps', '10', '--min-steps', '10',
    '--eval-every', '10', '--log-every', '1',
    '--eval-n', '2', '--diag', '0',
    '--ablation-every', '-1',
    '--save-every', '10', '--save-minutes', '5',
    '--tag', 'v3_colab_smoke'
]

print('Running 10-step Colab smoke...')
rc = subprocess.run(smoke, cwd=ROOT).returncode
print('Smoke exit code:', rc)
if rc != 0:
    raise RuntimeError('Smoke failed; do not start full training.')

## Full training

The checkpoint directory is persisted on Drive. Re-run this cell after a
disconnect; it finds the newest checkpoint and resumes.

`timeout 37800` stops before Colab's 12-hour limit. The training script itself
also saves atomically every 30 minutes.

In [ ]:
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
local_run = ROOT / 'runs' / RUN_DIR_NAME
if local_run.is_symlink() or local_run.exists():
    if local_run.is_symlink() or local_run.is_file():
        local_run.unlink()
    else:
        shutil.rmtree(local_run)
local_run.symlink_to(DRIVE_RUN, target_is_directory=True)

import shlex
full = common + [
    '--steps', '20000',
    '--min-steps', '4000',
    '--early-stop', '1',
    '--eval-every', '500',
    '--patience', '6',
    '--min-delta', '0.005',
    '--diag', '1', '--diag-n', '40',
    '--ablation-every', '-1',
    '--save-every', '500', '--save-minutes', '30',
    '--tag', RUN_TAG
]

ckpt_dir = DRIVE_RUN / 'ckpt'
steps = sorted(ckpt_dir.glob('step_*.pt')) if ckpt_dir.exists() else []
if steps:
    latest = steps[-1]
    full += ['--resume', str(latest)]
    print('Resuming from:', latest)
else:
    print('Starting a new full run.')

log_path = DRIVE_RUN / 'train.log'
quoted = ' '.join(shlex.quote(x) for x in full)
shell_cmd = (
    'set -o pipefail; '
    'timeout --signal=TERM 37800 '
    + quoted
    + ' 2>&1 | tee -a ' + shlex.quote(str(log_path))
)
print(shell_cmd)
rc = subprocess.run(['bash', '-lc', shell_cmd], cwd=ROOT).returncode
print('Full-run segment exit code:', rc)
print('124 or 143 means the session time guard stopped it; rerun to resume.')

In [ ]:
from pathlib import Path
ckpts = sorted((DRIVE_RUN / 'ckpt').glob('step_*.pt'))
print('Persistent run:', DRIVE_RUN)
print('Checkpoint count:', len(ckpts))
if ckpts:
    p = ckpts[-1]
    print('Latest:', p.name, f'{p.stat().st_size / 2**30:.2f} GiB', p.stat().st_mtime)
log = DRIVE_RUN / 'train.log'
if log.exists():
    print('\nLog tail:')
    print(''.join(log.read_text(encoding='utf-8', errors='replace').splitlines(True)[-80:]))